In [1]:
import numpy as np
import pandas as pd
from src import Replicator

In [2]:
methods = ['cs', '4p']
estimators = ['x', 'mu', 'var']

In [13]:
def repeatCompare(fn, R, n, param, tau, cstr=np.array([10]*2), sclr=1):
    rslt = {}
    threads = {key: {} for key in methods}
    for key in methods:
        for i in range(5):
            name = '{}#{}#'.format(key,i+1)
            threads[key][name] = Replicator(fn, rslt, key, R, n, param[key], tau, cstr, sclr, name)
    
    for key in methods:
        for name in threads[key]:
            threads[key][name].start()

    for key in methods:
        for name in threads[key]:
            threads[key][name].join()

    return rslt

# Objective Functions

In [4]:
L = np.array([
    [2, 0],
    [-1, 2]
])
A = L @ L.T
x = np.array([1, -2])
b = A @ x
c = 20
y = x @ A @ x / 2 - b @ x + c

def mu_fn(x, c=0): return x @ A @ x / 2 - b @ x + c

def sigmoid(y): return 1 / (1 + np.exp(-y))

## Continuous

In [5]:
def fn_c1(x, tau=1):
    mu = mu_fn(x, c=20)
    sclr = 1 + np.linalg.norm(x) * np.cos(np.pi * np.linalg.norm(x))
    noise = np.random.normal(size=tau)
    return mu + sclr * np.mean(noise)

def fn_c2(x, tau=1):
    mu = 20 * sigmoid(mu_fn(x)/800 - 2)
    sclr = 1 + np.linalg.norm(x) * np.cos(np.pi * np.linalg.norm(x))
    noise = np.random.normal(size=tau)
    return mu + sclr * np.mean(noise)

## Discrete/Binary

In [6]:
def fn_b1(x, tau=1):
    p = min(.99, max(.01, mu_fn(x)/2000 + .3))
    return np.random.binomial(tau, p) / tau

def fn_b2(x, tau=1):
    p = min(.99, max(.01, 2 * sigmoid(mu_fn(x)/600 - 2)))
    return np.random.binomial(tau, p) / tau

# Preliminaries
## Parameters

In [7]:
R = 25; n = int(1e5)
param = {'cs': .05, '4p': 3}
tau = 18, 2
cstr = np.array([10]*2)
sclr = 100

In [8]:
fn = {
    'c1': fn_c1,
    'c2': fn_c2,
    'b1': fn_b1,
    'b2': fn_b2
}
# methods = ['cs', '4p']
# estimators = ['x', 'mu', 'var']

# Main

In [12]:
for ftp in fn:
    cache = repeatCompare(fn[ftp], R, n, param, tau, cstr, sclr)
    for key in methods:
        for each in estimators:
            cache[key][each].to_csv("./output/d2Chp#A#{}_{}_{}.csv".format(ftp, key, each), index=False)

Thread cs#7# finishes replication 1 and takes 500954.55 ms
	final estimation x=(0.9952418997261869, -2.065435938763098), mu=3.98 and var=113.00
Thread cs#3# finishes replication 1 and takes 501024.24 ms
	final estimation x=(0.9892609810563615, -1.9742973406533422), mu=4.05 and var=101.46
Thread cs#4# finishes replication 1 and takes 501099.91 ms
	final estimation x=(1.103841308859971, -2.0210206546553353), mu=4.00 and var=119.99
Thread cs#1# finishes replication 1 and takes 501226.35 ms
	final estimation x=(1.0038585210162656, -2.006300430212306), mu=3.92 and var=95.59
Thread cs#6# finishes replication 1 and takes 501262.19 ms
	final estimation x=(1.0223512137339175, -1.9627432742900575), mu=4.08 and var=122.48
Thread cs#5# finishes replication 1 and takes 501444.40 ms
	final estimation x=(1.0231320726638904, -1.9225587026896194), mu=3.95 and var=107.69
Thread cs#2# finishes replication 1 and takes 501492.18 ms
	final estimation x=(1.0107135979953283, -2.071335959486948), mu=4.04 and v

In [9]:
R = 25; n = int(1e4)
param = {'cs': .05, '4p': 3}
tau = 18, 2
cstr = np.array([10]*2)
sclr = 1000

for ftp in ['b1', 'b2']:
    cache = repeatCompare(fn[ftp], R, n, param, tau, cstr, sclr)
    for key in methods:
        for each in estimators:
            cache[key][each].to_csv("./output/L/d2Chp#L#{}_{}_{}.csv".format(ftp, key, each), index=False)

Thread cs#6# finishes replication 1 and takes 29295.85 ms
	final estimation x=(6.413033351734591, 0.2342231166356934), mu=0.31 and var=0.01
Thread cs#3# finishes replication 1 and takes 29349.61 ms
	final estimation x=(2.7127867075616625, -1.719547212327673), mu=0.28 and var=0.01
Thread cs#2# finishes replication 1 and takes 29391.43 ms
	final estimation x=(-2.4415508456623343, -5.683352344868926), mu=0.29 and var=0.01
Thread cs#4# finishes replication 1 and takes 29407.36 ms
	final estimation x=(6.218670111198375, -1.2281247090028549), mu=0.30 and var=0.01
Thread cs#7# finishes replication 1 and takes 29609.47 ms
	final estimation x=(3.991466458486992, -0.36107195735604924), mu=0.30 and var=0.01
Thread cs#5# finishes replication 1 and takes 29687.12 ms
	final estimation x=(-0.07942590766911171, 0.10739151041499752), mu=0.31 and var=0.01
Thread cs#1# finishes replication 1 and takes 29971.87 ms
	final estimation x=(5.956702115869842, 0.2908673533020059), mu=0.32 and var=0.01
Thread 4p#

In [14]:
R = 10; n = int(1e5)
param = {'cs': .05, '4p': 3}
tau = 18, 2
cstr = np.array([10]*2)
sclr = 1000

for ftp in ['b1']:
    cache = repeatCompare(fn[ftp], R, n, param, tau, cstr, sclr)
    for key in methods:
        for each in estimators:
            cache[key][each].to_csv("./output/b1chp4p/d2Chp#{}_{}_{}.csv".format(ftp, key, each), index=False)

Thread cs#5# finishes replication 1 and takes 231616.92 ms
	final estimation x=(0.9537929488571004, -3.6234893179594785), mu=0.28 and var=0.01
Thread cs#4# finishes replication 1 and takes 231773.23 ms
	final estimation x=(1.1520102756938002, -2.482818120452854), mu=0.31 and var=0.01
Thread cs#3# finishes replication 1 and takes 232022.13 ms
	final estimation x=(1.1801181371659977, -0.7924957107603756), mu=0.30 and var=0.01
Thread cs#2# finishes replication 1 and takes 232053.99 ms
	final estimation x=(0.08594260065818078, -1.834875651170186), mu=0.29 and var=0.01
Thread cs#1# finishes replication 1 and takes 232226.23 ms
	final estimation x=(0.5517501840606632, -3.630550835274221), mu=0.31 and var=0.01
Thread 4p#3# finishes replication 1 and takes 363618.00 ms
	final estimation x=(-2.33825531291633, -4.96012834762856), mu=0.32 and var=0.01
Thread 4p#2# finishes replication 1 and takes 364124.76 ms
	final estimation x=(2.011983214924605, -1.2175903026764325), mu=0.30 and var=0.01
Threa